In [41]:
# ── Chargement et merge des 4 fichiers CSV ──────────────────────
# ── Chargement et merge des USAGERS (2021-2024) ──────────────────────────
# Séparateurs : 2021-23 (;) | 2024 (,)
# Harmonisation : 2022 Accident_Id -> Num_Acc + Typage String (anti-float)
# Analyse : Ajout colonne 'annee' 

import pandas as pd
import numpy as np

df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - caract 2021.csv', sep=';', dtype={'Num_Acc': str})
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - caract 2022.csv', sep=';')
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - caract 2023.csv', sep=';', dtype={'Num_Acc': str})
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - caract 2024.csv', sep=',',  dtype={'Num_Acc': str})

# Correction 2022 : renommer Accident_Id → Num_Acc AVANT le concat
df_2022 = df_2022.rename(columns={'Accident_Id': 'Num_Acc'})
df_2022['Num_Acc'] = df_2022['Num_Acc'].astype(str)

df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

caract_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)

print(f"✅ Merge OK — {caract_df.shape[0]:,} lignes x {caract_df.shape[1]} colonnes")
print(f"NaN Num_Acc : {caract_df['Num_Acc'].isna().sum()}")
print(f"NaN Num_Acc : {caract_df['Num_Acc'].isna().sum()}")

✅ Merge OK — 221,044 lignes x 16 colonnes
NaN Num_Acc : 0
NaN Num_Acc : 0


In [42]:
# ── Création de la colonne date ─────────────────────────────────
h_fix = caract_df['hrmn'].astype(str).str.replace(':', '').str.replace('nan', '0000').str.zfill(4)

date_string = (
    caract_df['an'].astype(str) + '-' +
    caract_df['mois'].astype(str).str.zfill(2) + '-' +
    caract_df['jour'].astype(str).str.zfill(2) + ' ' +
    h_fix.str[:2] + ':' + h_fix.str[2:]
)

caract_df['date'] = pd.to_datetime(date_string, errors='coerce')

errors = caract_df['date'].isna().sum()
print(f"Dates invalides (NaT) : {errors}")
print(f"✅ Colonne date créée")
caract_df[['an', 'mois', 'jour', 'hrmn', 'date']].head()

Dates invalides (NaT) : 0
✅ Colonne date créée


,an,mois,jour,hrmn,date
0,2024,3,25,7:40,2024-03-25 07:40:00
1,2024,3,20,15:05,2024-03-20 15:05:00
2,2024,3,22,19:30,2024-03-22 19:30:00
3,2024,3,24,17:50,2024-03-24 17:50:00
4,2024,3,25,19:35,2024-03-25 19:35:00


In [43]:
# ── Suppression des colonnes remplacées ou inutiles ──────────────
caract_df = caract_df.drop(columns=['an', 'mois', 'jour', 'hrmn', 'adr', 'Accident_Id'],
                            errors='ignore')
# errors='ignore' évite une erreur si une colonne n'existe pas dans certaines années

print(f"✅ Colonnes supprimées")
print(f"Colonnes restantes : {list(caract_df.columns)}")

✅ Colonnes supprimées
Colonnes restantes : ['Num_Acc', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'lat', 'long', 'annee', 'date']


In [44]:
# ── Renommage colonnes caractéristiques ──────────────────────────
caract_df = caract_df.rename(columns={
    'lum'           : 'luminosite',
    'dep'           : 'departement',
    'com'           : 'commune',
    'agg'           : 'agglomeration',
    'int'           : 'intersection',
    'atm'           : 'meteo',
    'col'           : 'type_collision',
    'lat'           : 'latitude',
    'long'          : 'longitude',
})
print(f"✅ Colonnes renommées : {list(caract_df.columns)}")

✅ Colonnes renommées : ['Num_Acc', 'luminosite', 'departement', 'commune', 'agglomeration', 'intersection', 'meteo', 'type_collision', 'latitude', 'longitude', 'annee', 'date']


In [45]:
# ── Remplacement des -1 par NaN ─────────────────────────────────

cols_sentinel = ['luminosite', 'intersection', 'meteo', 'type_collision']


caract_df[cols_sentinel] = caract_df[cols_sentinel].replace(-1, np.nan)


print("✅ Valeurs -1 remplacées par NaN")
print(caract_df[cols_sentinel].isna().sum())


✅ Valeurs -1 remplacées par NaN
luminosite          4
intersection       13
meteo              13
type_collision    102
dtype: int64


In [46]:
# ── Correction des coordonnées GPS ──────────────────────────────

caract_df['latitude']  = caract_df['latitude'].astype(str).str.replace(',', '.').pipe(pd.to_numeric, errors='coerce')
caract_df['longitude'] = caract_df['longitude'].astype(str).str.replace(',', '.').pipe(pd.to_numeric, errors='coerce')

print(f"✅ latitude/longitude corrigées")
# Utilise bien les nouveaux noms ici :
print(f"latitude  — Min: {caract_df['latitude'].min():.2f}  Max: {caract_df['latitude'].max():.2f}  NaN: {caract_df['latitude'].isna().sum()}")
print(f"longitude — Min: {caract_df['longitude'].min():.2f}  Max: {caract_df['longitude'].max():.2f}  NaN: {caract_df['longitude'].isna().sum()}")

✅ latitude/longitude corrigées
latitude  — Min: -23.40  Max: 51.08  NaN: 0
longitude — Min: -178.09  Max: 168.11  NaN: 0


In [47]:
# ── Renommage colonnes caractéristiques ──────────────────────────
caract_df = caract_df.rename(columns={
    'lum'           : 'luminosite',
    'dep'           : 'departement',
    'com'           : 'commune',
    'agg'           : 'agglomeration',
    'int'           : 'intersection',
    'atm'           : 'meteo',
    'col'           : 'type_collision',
    'lat'           : 'latitude',
    'long'          : 'longitude',
})
print(f"✅ Colonnes renommées : {list(caract_df.columns)}")

✅ Colonnes renommées : ['Num_Acc', 'luminosite', 'departement', 'commune', 'agglomeration', 'intersection', 'meteo', 'type_collision', 'latitude', 'longitude', 'annee', 'date']


In [48]:
# ── Vérification finale avant export ────────────────────────────

print(f"Shape final : {caract_df.shape}")

print("\n--- Types des colonnes ---")
print(caract_df.dtypes)

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': caract_df.isna().sum(),
    'NaN %': (caract_df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print("\n--- Vérif aucun -1 restant ---")
for col in caract_df.select_dtypes(include='number').columns:
    n = (caract_df[col] == -1).sum()
    if n > 0:
        print(f"  ⚠️  {col} : {n} valeurs -1 restantes")
print("✅ Aucun -1 restant")

print("\n--- Doublons ---")
print(f"Doublons Num_Acc : {caract_df.duplicated('Num_Acc').sum()}")

Shape final : (221044, 12)

--- Types des colonnes ---
Num_Acc                   object
luminosite               float64
departement               object
commune                   object
agglomeration              int64
intersection             float64
meteo                    float64
type_collision           float64
latitude                 float64
longitude                float64
annee                      int64
date              datetime64[ns]
dtype: object

--- NaN par colonne ---
                NaN count  NaN %
luminosite              4    0.0
intersection           13    0.0
meteo                  13    0.0
type_collision        102    0.0

--- Vérif aucun -1 restant ---
  ⚠️  longitude : 1 valeurs -1 restantes
✅ Aucun -1 restant

--- Doublons ---
Doublons Num_Acc : 0


In [49]:
# ── Export du fichier nettoyé ───────────────────────────────────
# Export dans le dossier Dataset avec séparateur point-virgule
# Ce fichier sera utilisé pour la jointure avec les tables usagers, lieux, vehicules

caract_df.to_csv('Dataset/caract_2021_2024_clean.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Exporté — {len(caract_df):,} lignes x {caract_df.shape[1]} colonnes")

✅ Exporté — 221,044 lignes x 12 colonnes
